# 03 — Tratamento / Silver
Inclui engenharia temporal, alvo e 12 indicadores para os múltiplos valores de traçado.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
for p in [DATA/"raw", DATA/"processed", OUT/"auditoria", OUT/"eda", OUT/"estatistica", OUT/"modelos", OUT/"interpretabilidade", OUT/"modelo_final"]:
    p.mkdir(parents=True,exist_ok=True)
SILVER = DATA/"processed"/"prf_acidentes_silver_2020_2025.parquet"
import re,unicodedata
df=pd.read_parquet(DATA/"processed"/"prf_acidentes_2020_2025.parquet")
df["data_inversa"]=pd.to_datetime(df.data_inversa,errors="coerce")
df["ano"]=df.data_inversa.dt.year
df["mes"]=df.data_inversa.dt.month
df["trimestre"]=df.data_inversa.dt.quarter
df["dia_mes"]=df.data_inversa.dt.day
df["dia_ano"]=df.data_inversa.dt.dayofyear
df["fim_semana"]=df.data_inversa.dt.dayofweek.isin([5,6]).astype("int8")
h=pd.to_datetime(df.horario.astype("string"),format="%H:%M:%S",errors="coerce")
df["hora"]=h.dt.hour; df["minuto"]=h.dt.minute
df["faixa_horaria"]=pd.cut(df.hora,bins=[-1,5,11,17,23],labels=["Madrugada","Manhã","Tarde","Noite"]).astype("string").fillna("Não informado")
for c in ["mortos","feridos_graves","feridos_leves","feridos","veiculos","km","pessoas"]:
    df[c]=pd.to_numeric(df[c],errors="coerce")
for c in ["tipo_acidente","classificacao_acidente","regional","delegacia","uop"]:
    df[c]=df[c].fillna("Não informado")
df["acidente_grave"]=((df.mortos>0)|(df.feridos_graves>0)).astype("int8")
df["acidente_fatal"]=(df.mortos>0).astype("int8")

In [ ]:
def slug(s):
    s=unicodedata.normalize("NFKD",str(s)).encode("ascii","ignore").decode("ascii")
    return re.sub("[^a-z0-9]+","_",s.lower()).strip("_")
tr=df.tracado_via.fillna("").astype(str).str.get_dummies(sep=";")
tr.columns=["tracado_"+slug(c) for c in tr.columns]
df=pd.concat([df,tr.astype("int8")],axis=1)
cols=[c for c in df if c.startswith("tracado_") and c!="tracado_via"]
df["qtd_caracteristicas_tracado"]=df[cols].sum(axis=1)
assert df.id.is_unique
df.to_parquet(SILVER,index=False)
print("Base Silver:",df.shape,"Classe grave:",df.acidente_grave.mean())